# Day 2: Market Data Acquisition & Stochastic Process Simulation
**Prepared by:** Aditya (Lead), Anhum (Data), Tanveer (EDA & Verification)

This interactive notebook demonstrates:
1. Ingestion and local caching of real-world multi-asset market data (`AAPL`, `MSFT`, `SPY`).
2. Simulation of Geometric Brownian Motion (GBM) vs Merton Jump-Diffusion models.
3. Comparative statistical analysis of stylized facts (Volatility, Skewness, Kurtosis, and Fat Tails).

In [ ]:
import sys
sys.path.append("..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data.synthetic_generator import StochasticProcessGenerator
from src.data.real_data_loader import RealMarketDataLoader
from src.data.inspector import DataInspector
from src.utils.finance_math import calculate_log_returns

sns.set_theme(style="whitegrid")
print("Modules successfully imported.")

## 1. Simulate Stochastic Processes: GBM vs. Merton Jump Diffusion

In [ ]:
synth = StochasticProcessGenerator(random_seed=42)

# Simulate 1512 trading days (~6 years)
gbm_path = synth.generate_gbm(s0=100.0, mu=0.08, sigma=0.20, n_days=1512)
merton_path = synth.generate_merton_jump(
    s0=100.0, mu=0.08, sigma=0.18, lambda_j=0.25, mu_j=-0.08, sigma_j=0.12, n_days=1512
)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(gbm_path.values, label="Geometric Brownian Motion (Continuous)", color="#1f77b4", lw=1.5)
ax.plot(merton_path.values, label="Merton Jump Diffusion (Discontinuous Jumps)", color="#d62728", lw=1.5)
ax.set_title("Synthetic Price Trajectories: GBM vs. Merton Jump Diffusion", fontsize=13, fontweight="bold")
ax.set_xlabel("Trading Days")
ax.set_ylabel("Simulated Price ($)")
ax.legend(frameon=True)
plt.show()

## 2. Load Real-World Market Data (Cached from `data/raw`)

In [ ]:
loader = RealMarketDataLoader(cache_dir="../data/raw", use_cache=True)
aapl_df = loader.load_data("AAPL", "2018-01-01", "2024-12-31")
spy_df = loader.load_data("SPY", "2018-01-01", "2024-12-31")

print(f"AAPL loaded: {len(aapl_df)} trading bars from {aapl_df.index.min().date()} to {aapl_df.index.max().date()}")
print(f"SPY loaded:  {len(spy_df)} trading bars from {spy_df.index.min().date()} to {spy_df.index.max().date()}")
aapl_df.head()

## 3. Stylized Facts Comparison: Real vs. Synthetic Distributions
Notice how real market returns and Merton Jump Diffusion exhibit significant **excess kurtosis** ($> 3.0$), capturing fat tails that pure Gaussian GBM completely misses.

In [ ]:
comparison_table = DataInspector.compare_real_vs_synthetic(
    real_series=aapl_df["Adj Close"],
    gbm_series=gbm_path,
    merton_series=merton_path
)
comparison_table

## 4. Return Distribution Visualization (Fat Tails & Leptokurtosis)

In [ ]:
r_aapl = calculate_log_returns(aapl_df["Adj Close"])
r_gbm = calculate_log_returns(gbm_path)
r_merton = calculate_log_returns(merton_path)

fig, axes = plt.subplots(1, 3, figsize=(16, 4), sharey=True)

sns.histplot(r_aapl, kde=True, ax=axes[0], color="#2ca02c", bins=50)
axes[0].set_title(f"Real Market (AAPL)\nKurtosis: {comparison_table.loc['Real Market Asset', 'kurtosis']:.2f}")
axes[0].set_xlabel("Daily Log Return")

sns.histplot(r_gbm, kde=True, ax=axes[1], color="#1f77b4", bins=50)
axes[1].set_title(f"Synthetic GBM\nKurtosis: {comparison_table.loc['Synthetic GBM (Continuous)', 'kurtosis']:.2f}")
axes[1].set_xlabel("Daily Log Return")

sns.histplot(r_merton, kde=True, ax=axes[2], color="#d62728", bins=50)
axes[2].set_title(f"Synthetic Merton Jump\nKurtosis: {comparison_table.loc['Synthetic Merton Jump (Discontinuous)', 'kurtosis']:.2f}")
axes[2].set_xlabel("Daily Log Return")

plt.tight_layout()
plt.show()